In [1]:
import asyncio
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy, TimeoutPolicy, Command
from langgraph.errors import NodeError

# ==========================================
# 1. 定义状态 (State)
# ==========================================
class WorkflowState(TypedDict):
    text: str
    status: str
    logs: list[str]  # 用于记录执行过程，方便观察容错机制的触发顺序

# ==========================================
# 2. 定义节点函数 (必须是 async，因为 Timeout 仅支持异步节点)
# ==========================================
async def fetch_text(state: WorkflowState) -> WorkflowState:
    logs = state.get("logs", []) + ["[1. fetch_text] 成功获取原始文本。"]
    return {"text": "Hello LangGraph", "status": "fetched", "logs": logs}

In [2]:
async def translate_text(state: WorkflowState) -> WorkflowState:
    logs = state.get("logs", []) + ["[2. translate_text] 开始调用外部翻译 API..."]
    
    # 模拟故障：API 响应极慢，耗时 2 秒
    await asyncio.sleep(2) 
    
    logs.append("[2. translate_text] 翻译成功。")
    return {"text": "你好 LangGraph", "status": "translated", "logs": logs}

In [3]:
async def fallback_translate(state: WorkflowState) -> WorkflowState:
    logs = state.get("logs", []) + ["[4. fallback_translate] 触发降级方案：使用本地简易规则翻译。"]
    return {"text": "[本地降级] Hello LangGraph", "status": "fallback_success", "logs": logs}

In [4]:
# 定义错误处理机制
async def handle_error(state: WorkflowState, error: NodeError) -> Command:
    logs = state.get("logs", []) + ["[3. handle_error] 捕获到错误：{}".format(error)]
    return Command(
        update={"status":"错误情况下的兜底处理机制！", "logs": logs},
        goto="fallback_translate"
    )

In [5]:
builder = StateGraph(WorkflowState)
builder.set_node_defaults(
    retry_policy=RetryPolicy(max_attempts=2), # 默认最多尝试 2 次
)
builder.add_node("fetch_text", fetch_text)
builder.add_node("fallback_translate", fallback_translate)
builder.add_node(
    "translate_text",
    translate_text,
    # [超时控制] 硬性限制：单次尝试最多运行 1 秒 (NodeTimeoutError)
    timeout=TimeoutPolicy(run_timeout=1.0),
    # [重试机制] 针对该节点定制：最多尝试 2 次，首次重试等待 0.1 秒
    retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.1),
    # [错误处理] 重试耗尽后，执行专用的补偿逻辑
    error_handler=handle_error,
)

In [6]:
builder.add_edge(START, "fetch_text")
builder.add_edge("fetch_text", "translate_text")
builder.add_edge("translate_text", END)
builder.add_edge("fallback_translate", END)

graph = builder.compile()


In [10]:
result = await graph.ainvoke({"text":"Hello LangGraph", "status": "start", "logs": []})

c:\Users\21566\anaconda3\envs\ai-langchain1.0\Lib\site-packages\langsmith\_internal\_agent_addressing.py:61: LangSmithBetaWarning: Agent addressing (`agent_id` / `agent_environment`) is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


In [11]:
result

{'text': '[本地降级] Hello LangGraph',
 'status': 'fallback_success',
 'logs': ['[1. fetch_text] 成功获取原始文本。',
  '[3. handle_error] 捕获到错误：NodeError(node=\'translate_text\', error=NodeTimeoutError("Node \'translate_text\' exceeded its run timeout of 1.000s (elapsed: 1.016s)."))',
  '[4. fallback_translate] 触发降级方案：使用本地简易规则翻译。']}